# PyTorch ile Derin Öğrenme · Diziler: RNN, LSTM ve dikkat

Macademia · Büyükler için

> **Colab'da çalıştırma:** Bu dosyayı [colab.research.google.com](https://colab.research.google.com) adresinde **Dosya → Not defteri yükle** ile aç. GPU için **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç. Hücreleri yukarıdan aşağı sırayla çalıştır (Shift + Enter). PyTorch Colab'da kuruludur; bilgisayarında çalıştırmak istersen `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # tekrarlanabilirlik
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Görev: dizinin başındaki işareti hatırla
Dizinin ilk adımındaki değerin işareti etiketi belirler; arası gürültü. Uzun dizilerde RNN zorlanır.

In [ ]:
from torch import nn
def veri(n, T):
    x = torch.randn(n, T, 1) * 0.5
    y = (x[:, 0, 0] > 0).long()
    x[:, 0, 0] = torch.where(y == 1, 1.0, -1.0)
    return x, y
class Model(nn.Module):
    def __init__(self, kind='lstm', h=32):
        super().__init__()
        self.rnn = (nn.LSTM if kind == 'lstm' else nn.RNN)(1, h, batch_first=True); self.fc = nn.Linear(h, 2)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1])

## 2. RNN ve LSTM karşılaştırması

In [ ]:
def egit(kind, T, steps=300):
    torch.manual_seed(0); m = Model(kind).to(device); opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    for _ in range(steps):
        x, y = veri(64, T); x, y = x.to(device), y.to(device)
        opt.zero_grad(); l = nn.functional.cross_entropy(m(x), y); l.backward()
        nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    x, y = veri(500, T)
    with torch.no_grad():
        return (m(x.to(device)).argmax(1).cpu() == y).float().mean().item()
for T in [10, 50, 100]:
    print(f'T={T:3d}  RNN {egit("rnn", T):.2f}  LSTM {egit("lstm", T):.2f}')

## 3. Ölçekli nokta çarpımı dikkati

In [ ]:
import torch.nn.functional as F
Q = K_ = V = torch.randn(1, 5, 16)
w = torch.softmax(Q @ K_.transpose(-2, -1) / 16 ** 0.5, dim=-1)
print(w.sum(-1))
print(torch.allclose(w @ V, F.scaled_dot_product_attention(Q, K_, V), atol=1e-6))
mha = nn.MultiheadAttention(16, num_heads=4, batch_first=True)
out, att = mha(Q, K_, V)
print(out.shape, att.shape)

## Alıştırmalar
1. `nn.GRU` ile aynı karşılaştırmayı yap.
2. Gradyan kırpmayı kaldır; RNN'de ne oldu?
3. Dikkat ağırlıklarını ısı haritası olarak çiz.